In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split


DATA_DIR = './' 

if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Используем видеокарту NVIDIA (CUDA)")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    print("ВНИМАНИЕ: Видеокарта не обнаружена. Обучение пойдет на CPU и будет ОЧЕНЬ медленным!")

# 3. Загрузка таблиц
labels_df = pd.read_csv(os.path.join(DATA_DIR, 'labels.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f"Загружено строк в labels: {len(labels_df)}")

Используем видеокарту NVIDIA (CUDA)
Загружено строк в labels: 10222


In [2]:
breeds = list(sample_sub.columns[1:])
breed_to_idx = {breed: idx for idx, breed in enumerate(breeds)}

# Добавляем полные пути к картинкам и индексы классов
labels_df['img_path'] = labels_df['id'].apply(lambda x: os.path.join(DATA_DIR, 'train', f"{x}.jpg"))
labels_df['label_idx'] = labels_df['breed'].map(breed_to_idx)

# Разделим данные на train (90%) и validation (10%) для контроля процесса
train_df, val_df = train_test_split(
    labels_df, 
    test_size=0.1, 
    random_state=42, 
    stratify=labels_df['label_idx']
)

# Подготовим список тестовых изображений на основе шаблона
test_df = pd.DataFrame({
    'id': sample_sub['id'],
    'img_path': sample_sub['id'].apply(lambda x: os.path.join(DATA_DIR, 'test', f"{x}.jpg"))
})

# ==========================================
# 3. КЛАСС ДЛЯ РАБОТЫ С ДАТАСЕТОМ
# ==========================================
class DogDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df
        self.transform = transform
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['img_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, row['id']
        return image, row['label_idx']

# Простые и быстрые трансформации (размер 224x224 оптимален для быстрого старта)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Легкая аугментация, чтобы модель меньше переобучалась
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# DataLoader'ы (если памяти мало или выдает ошибку, уменьшите batch_size до 16 или 8)
BATCH_SIZE = 32

train_loader = DataLoader(DogDataset(train_df, transform=train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(DogDataset(val_df, transform=val_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(DogDataset(test_df, transform=val_transforms, is_test=True), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ==========================================
# 4. СОЗДАНИЕ И НАСТРОЙКА МОДЕЛИ
# ==========================================
# Загружаем ResNet18 с предобученными весами ImageNet
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Для быстрого первого сабмишена заморозим все веса, кроме последнего слоя
for param in model.parameters():
    param.requires_grad = False

# Заменяем финальный классификатор под наши 120 классов
model.fc = nn.Linear(model.fc.in_features, 120)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Обучаем только веса слоя fc
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

# ==========================================
# 5. ЦИКЛ ОБУЧЕНИЯ (2 ЭПОХИ)
# ==========================================
EPOCHS = 2  # Для проверки пайплайна достаточно пары эпох

print("Начало обучения базовой модели...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    # Быстрая валидация
    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            
    epoch_loss = running_loss / len(train_df)
    epoch_val_loss = val_loss / len(val_df)
    epoch_acc = correct.double() / len(val_df)
    print(f"-> Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {epoch_acc:.4f}\n")

# ==========================================
# 6. ГЕНЕРАЦИЯ ПРЕДСКАЗАНИЙ ДЛЯ ТЕСТА
# ==========================================
print("Генерируем предсказания для тестовой выборки...")
model.eval()
test_preds = []
test_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Тестирование"):
        images = images.to(device)
        outputs = model(images)
        
        # Переводим логиты модели в вероятности
        probs = torch.softmax(outputs, dim=1)
        test_preds.append(probs.cpu().numpy())
        test_ids.extend(ids)

# Объединяем предсказания в одну матрицу
test_preds = np.vstack(test_preds)

# Создаем финальный DataFrame
submission_df = pd.DataFrame(test_preds, columns=breeds)
submission_df.insert(0, 'id', test_ids)

# Сохраняем результат на ваш компьютер
submission_df.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно создан! Вы можете загрузить его на Kaggle.")

Начало обучения базовой модели...


Эпоха 1/2: 100%|██████████| 288/288 [00:41<00:00,  7.00it/s]


-> Train Loss: 2.6215 | Val Loss: 1.4503 | Val Accuracy: 0.6569



Эпоха 2/2: 100%|██████████| 288/288 [00:37<00:00,  7.76it/s]


-> Train Loss: 1.2132 | Val Loss: 1.0904 | Val Accuracy: 0.7077

Генерируем предсказания для тестовой выборки...


Тестирование: 100%|██████████| 324/324 [00:45<00:00,  7.20it/s]


Файл submission.csv успешно создан! Вы можете загрузить его на Kaggle.


In [3]:
# =====================================================================
# НОВАЯ УЛУЧШЕННАЯ МОДЕЛЬ (EFFICIENTNET-B0) И ГЕНЕРАЦИЯ САБМИШЕНА
# =====================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import pandas as pd
from torchvision import models

print(f"Используемое устройство: {device}")

# 1. Загружаем предобученную модель EfficientNet-B0
# Она эффективнее и легче, чем ResNet18, что критично для CPU
model_improved = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# 2. Замораживаем веса базовой сети (экстрактора признаков)
for param in model_improved.parameters():
    param.requires_grad = False

# 3. Заменяем классификатор (добавляем Dropout для борьбы с переобучением)
in_features = model_improved.classifier[1].in_features
model_improved.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 120)
)

model_improved = model_improved.to(device)

# 4. Настройка функции потерь и оптимизатора AdamW
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_improved.classifier.parameters(), lr=1e-3, weight_decay=1e-2)

# 5. Цикл обучения (3 эпохи для баланса времени на CPU и качества)
EPOCHS = 3
print("Начало обучения улучшенной модели...")

for epoch in range(EPOCHS):
    model_improved.train()
    running_loss = 0.0
    
    for images, labels in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_improved(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    # Валидация в конце эпохи
    model_improved.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_improved(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            
    epoch_loss = running_loss / len(train_df)
    epoch_val_loss = val_loss / len(val_df)
    epoch_acc = correct.double() / len(val_df)
    print(f"-> Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {epoch_acc:.4f}\n")

# =====================================================================
# 6. ГЕНЕРАЦИЯ УЛУЧШЕННОГО ПРЕДСКАЗАНИЯ ДЛЯ ТЕСТА
# =====================================================================
print("Генерируем предсказания для тестовой выборки...")
model_improved.eval()
test_preds = []
test_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Тестирование"):
        images = images.to(device)
        outputs = model_improved(images)
        
        # Получаем вероятности классов с помощью Softmax
        probs = torch.softmax(outputs, dim=1)
        test_preds.append(probs.cpu().numpy())
        test_ids.extend(ids)

# Объединяем предсказания
test_preds = np.vstack(test_preds)

# Создаем финальный DataFrame для Kaggle
submission_df = pd.DataFrame(test_preds, columns=breeds)
submission_df.insert(0, 'id', test_ids)

# Сохраняем файл
submission_df.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно перезаписан! Вы можете загрузить его на Kaggle.")

Используемое устройство: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/br41nd34d/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 9.67MB/s]


Начало обучения улучшенной модели...


Эпоха 1/3: 100%|██████████| 288/288 [00:38<00:00,  7.57it/s]


-> Train Loss: 3.0272 | Val Loss: 1.8256 | Val Accuracy: 0.6188



...

-> Train Loss: 1.6341 | Val Loss: 1.4335 | Val Accuracy: 0.6510



Эпоха 3/3: 100%|██████████| 288/288 [00:48<00:00,  5.96it/s]


-> Train Loss: 1.2642 | Val Loss: 1.3080 | Val Accuracy: 0.6628

Генерируем предсказания для тестовой выборки...


Тестирование: 100%|██████████| 324/324 [00:40<00:00,  8.04it/s]


Файл submission.csv успешно перезаписан! Вы можете загрузить его на Kaggle.


In [ ]:
# =====================================================================
# ВАРИАНТ НА БАЗЕ MOBILENET_V3_LARGE И ГЕНЕРАЦИЯ САБМИШЕНА
# =====================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import pandas as pd
from torchvision import models

print(f"Используемое устройство: {device}")

# 1. Загружаем предобученную модель MobileNetV3-Large
# Модель оптимизирована для CPU и работает быстрее аналогов
model_improved = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

# 2. Замораживаем веса базовой сети (экстрактора признаков)
for param in model_improved.parameters():
    param.requires_grad = False

# 3. Заменяем только последний классифицирующий слой
# В оригинальной MobileNetV3-Large классификатор состоит из:
# Linear(960 -> 1280) -> Hardswish -> Dropout -> Linear(1280 -> 1000)
# Нам нужно заменить последний слой на выход из 120 нейронов
model_improved.classifier[3] = nn.Linear(1280, 120)

model_improved = model_improved.to(device)

# 4. Настройка функции потерь и оптимизатора AdamW
criterion = nn.CrossEntropyLoss()
# Обучаем только параметры классификатора (включая измененный слой)
optimizer = optim.AdamW(model_improved.classifier.parameters(), lr=1e-3, weight_decay=1e-2)

# 5. Цикл обучения (3 эпохи)
EPOCHS = 3
print("Начало обучения модели MobileNetV3...")

for epoch in range(EPOCHS):
    model_improved.train()
    running_loss = 0.0
    
    for images, labels in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_improved(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    # Валидация в конце эпохи
    model_improved.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_improved(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            
    epoch_loss = running_loss / len(train_df)
    epoch_val_loss = val_loss / len(val_df)
    epoch_acc = correct.double() / len(val_df)
    print(f"-> Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {epoch_acc:.4f}\n")

# =====================================================================
# 6. ГЕНЕРАЦИЯ ПРЕДСКАЗАНИЙ ДЛЯ ТЕСТА
# =====================================================================
print("Генерируем предсказания для тестовой выборки...")
model_improved.eval()
test_preds = []
test_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Тестирование"):
        images = images.to(device)
        outputs = model_improved(images)
        
        # Переводим выходы модели в вероятности классов
        probs = torch.softmax(outputs, dim=1)
        test_preds.append(probs.cpu().numpy())
        test_ids.extend(ids)

# Объединяем предсказания в матрицу
test_preds = np.vstack(test_preds)

# Создаем финальный DataFrame
submission_df = pd.DataFrame(test_preds, columns=breeds)
submission_df.insert(0, 'id', test_ids)

# Сохраняем результат
submission_df.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно создан на базе MobileNetV3 и готов к загрузке!")